# Algoritmos de optimización - Trabajo Práctico<br>
Nombre y Apellidos: Inés Pastor Espinosa  <br>
Url: <br>
Google Colab: https://colab.research.google.com/drive/1W_MZBiv9-QEI4vluLd8C0MrX52Cwp0Re <br>
Problema:
>1. Sesiones de doblaje <br>


Descripción del problema:

Se precisa coordinar el doblaje de una película. Los actores del doblaje deben coincidir en
las tomas en las que sus personajes aparecen juntos en las diferentes tomas. Los actores de
doblaje cobran todos la misma cantidad por cada día que deben desplazarse hasta el
estudio de grabación independientemente del número de tomas que se graben. No es
posible grabar más de 6 tomas por día. El objetivo es planificar las sesiones por día de
manera que el gasto por los servicios de los actores de doblaje sea el menor posible. Los
datos son:

Número de actores: 10

Número de tomas : 30

Actores/Tomas : https://bit.ly/36D8IuK
- 1 indica que el actor participa en la toma
- 0 en caso contrario




                                        

#Modelo
- ¿Como represento el espacio de soluciones?

El problema consiste en planificar la grabación de 30 tomas, con un máximo de 6 tomas por día.

El  número mínimo teórico de días necesarios es:

 $N_D=\frac{30}{6}=5$ .


Una solución se representa mediante un vector:

 $x=(x_1,x_2....,x_30)$
 donde:

  $x_i ∈ {0,1,2,3,4}$.
  
  Es decir, cada componente $x_i$ indica el día en que se graba la toma $i$.



- ¿Cual es la función objetivo?

Los actores cobran por cada día que trabajan, independientemente del número de tomas realizadas ese día.

El objetico es minimizar el número total de actor-días.

Siendo:

· $A_{i}^{a}​=1$ si el actor $a$ participa en la toma $i$.

· $x_i$ día asignado a la toma $i$.

Definimos:

$y_{a,d} = \begin{cases}
1 \text{ si el actor a participa en alguna toma del día d}\\
0 \text{ si no participa}
\end{cases}$


La función objetivo es:

$f(x)=\sum_{d=0}^{4}∑_{a=1}^{10} y_{a,d}$

- ¿Como implemento las restricciones?
Las restricciones a tener en cuenta y sus implementaciones son:


1.   Una toma se empieza y acaba el mismo dia. (Cada toma se asigna a un único día $x_i$).

2.   Máximo 6 tomas por día. (Generar individuos con exactamente 6 tomas por día o
añadir penalizaiones en el fitness sinun día supera el límite de tomas).

3. Si dos actores aparecen juntos en una toma, deben estar el mismo día en esa toma(Implícitamente dentro de la primera restricción).

In [ ]:
import random
import numpy as np

# Semilla para reproducibilidad
random.seed(42)
np.random.seed(42)

# Matriz que indica qué actores participan en cada toma (1=participa, 0=no participa)
actores_tomas = np.array([
[1,1,1,1,1,0,0,0,0,0],
[0,0,1,1,1,0,0,0,0,0],
[0,1,0,0,1,0,1,0,0,0],
[1,1,0,0,0,0,1,1,0,0],
[0,1,0,1,0,0,0,1,0,0],
[1,1,0,1,1,0,0,0,0,0],
[1,1,0,1,1,0,0,0,0,0],
[1,1,0,0,0,1,0,0,0,0],
[1,1,0,1,0,0,0,0,0,0],
[1,1,0,0,0,1,0,0,1,0],
[1,1,1,0,1,0,0,1,0,0],
[1,1,1,1,0,1,0,0,0,0],
[1,0,0,1,1,0,0,0,0,0],
[1,0,1,0,0,1,0,0,0,0],
[1,1,0,0,0,0,1,0,0,0],
[0,0,0,1,0,0,0,0,0,1],
[1,0,1,0,0,0,0,0,0,0],
[0,0,1,0,0,1,0,0,0,0],
[1,0,1,0,0,0,0,0,0,0],
[1,0,1,1,1,0,0,0,0,0],
[0,0,0,0,0,1,0,1,0,0],
[1,1,1,1,0,0,0,0,0,0],
[1,0,1,0,0,0,0,0,0,0],
[0,0,1,0,0,1,0,0,0,0],
[1,1,0,1,0,0,0,0,0,1],
[1,0,1,0,1,0,0,0,1,0],
[0,0,0,1,1,0,0,0,0,0],
[1,0,0,1,0,0,0,0,0,0],
[1,0,0,0,1,1,0,0,0,0],
[1,0,0,1,0,0,0,0,0,0]
])

NUM_TOMAS = 30
NUM_ACTORES = 10
NUM_DIAS = 5
MAX_TOMAS_DIA = 6


# FUNCIÓN DE COSTE
def calcular_coste(planificacion):
    """
    Calcula el coste total de una planificación de tomas en días.
    Penaliza si se excede el máximo de tomas por día.
    """
    coste = 0
    penalizacion = 0
    for dia in range(NUM_DIAS):
        tomas_dia = [i for i in range(NUM_TOMAS) if planificacion[i] == dia]

        # Penalización si se excede el máximo de tomas por día
        if len(tomas_dia) > MAX_TOMAS_DIA:
            penalizacion += (len(tomas_dia) - MAX_TOMAS_DIA) * 200

        if len(tomas_dia) > 0:
            actores_presentes = np.zeros(NUM_ACTORES)
            for t in tomas_dia:
                actores_presentes = np.logical_or(actores_presentes, actores_tomas[t])
            coste += np.sum(actores_presentes)

    return coste + penalizacion


# CREAR PLANIFICACIÓN INICIAL
def crear_planificacion_inicial():
    """
    Crea una planificación inicial con 6 tomas por día, distribuidas aleatoriamente.
    """
    planificacion = []
    for d in range(NUM_DIAS):
        planificacion += [d] * MAX_TOMAS_DIA
    random.shuffle(planificacion)
    return np.array(planificacion)


# OPERADORES DE PLANIFICACIÓN
def intercambiar_tomas(planificacion, prob=0.03):
    """
    Intercambia días de dos tomas aleatorias de forma segura.
    """
    nueva = planificacion.copy()
    for i in range(NUM_TOMAS):
        if random.random() < prob:
            j = random.randint(0, NUM_TOMAS-1)
            nueva[i], nueva[j] = nueva[j], nueva[i]
    return nueva

def combinar_planificaciones(p1, p2):
    """
    Combina dos planificaciones mediante un punto de corte.
    """
    punto = random.randint(1, NUM_TOMAS-1)
    h1 = np.concatenate([p1[:punto], p2[punto:]])
    h2 = np.concatenate([p2[:punto], p1[punto:]])
    return h1, h2

def seleccion_mejor_torneo(poblacion, k=3):
    """
    Selecciona la mejor planificación de un subconjunto aleatorio (torneo).
    """
    candidatos = random.sample(poblacion, k)
    candidatos.sort(key=lambda x: calcular_coste(x))
    return candidatos[0]

# ALGORITMO DE PLANIFICACIÓN
def planificar_tomas():
    """
    Algoritmo de planificación basado en heurística evolutiva.
    Mantiene las mejores planificaciones y combina/intercambia tomas para generar nuevas.
    """
    TAM_POBLACION = 300
    NUM_GENERACIONES = 800

    poblacion = [crear_planificacion_inicial() for _ in range(TAM_POBLACION)]

    for gen in range(NUM_GENERACIONES):
        nueva_poblacion = []

        # Mantener las 3 mejores planificaciones (elitismo)
        mejores = sorted(poblacion, key=lambda x: calcular_coste(x))[:3]
        nueva_poblacion.extend(mejores)

        while len(nueva_poblacion) < TAM_POBLACION:
            p1 = seleccion_mejor_torneo(poblacion)
            p2 = seleccion_mejor_torneo(poblacion)
            h1, h2 = combinar_planificaciones(p1, p2)
            nueva_poblacion.append(intercambiar_tomas(h1))
            if len(nueva_poblacion) < TAM_POBLACION:
                nueva_poblacion.append(intercambiar_tomas(h2))

        poblacion = nueva_poblacion

        if gen % 50 == 0:
            print(f"Generación {gen} - Mejor coste: {calcular_coste(mejores[0])}")

    mejor_planificacion = min(poblacion, key=lambda x: calcular_coste(x))
    return mejor_planificacion


# EJECUCIÓN
solucion_final = planificar_tomas()
print("\nMejor planificación encontrada:")
print(solucion_final)
print("Coste total:", calcular_coste(solucion_final))

Generación 0 - Mejor coste: 35
Generación 50 - Mejor coste: 30
Generación 100 - Mejor coste: 30
Generación 150 - Mejor coste: 29
Generación 200 - Mejor coste: 29
Generación 250 - Mejor coste: 29
Generación 300 - Mejor coste: 29
Generación 350 - Mejor coste: 29
Generación 400 - Mejor coste: 29
Generación 450 - Mejor coste: 29
Generación 500 - Mejor coste: 29
Generación 550 - Mejor coste: 29
Generación 600 - Mejor coste: 29
Generación 650 - Mejor coste: 29
Generación 700 - Mejor coste: 29
Generación 750 - Mejor coste: 29

Mejor planificación encontrada:
[4 4 2 1 2 4 4 0 0 1 1 0 2 3 1 2 0 3 3 4 1 0 3 3 2 1 2 4 3 0]
Coste total: 29


#Análisis
- ¿Que complejidad tiene el problema?. Orden de complejidad y Contabilizar el espacio de soluciones

El problema es de tipo NP-hard ya que para cada toma existen varias posibles asignaciones de día y no existe un algoritmo polinómico que garantice encontrar siempre la solución óptima para cualquier número de tomas y actores.

De hecho, como trabajo previo, se utilizó un algoritmo de Programación Lineal Entera, con el cual se obtuvo el óptimo (27 actor-días), pero incluso con un volumen de datos relativamente pequeño, el tiempo de ejecución superó los 5 minutos. Si se incrementa el número de tomas o actores, la complejidad temporal aumentaríade manera exponencial, volviéndose inviable para taaños mayores.


Cada toma puede asignarse a cualquiera de los cinco días disponibles, y hay 30 tomas, por lo que el tamaño del espacio de soluciones es :

$5^{30}$

Por tanto, la complejidad temporal crece exponencialmente con el número de tomas, de forma que un aumente en este parámetro provoca un crecimiento muy rápido del tiempo de computación.

En cuanto a la complejidad espacial , considerando que hemos elegido un tamaño de la población (TAM) de 300 y que cada solución almacena la asignación de 30 tomas, necesitamos almacenar un total de:  

$300x30=9000$ valores

#Diseño
- ¿Que técnica utilizo? ¿Por qué?

El problema se ha abordado mediante una técnica heuristica inspirada en la evolución natural, el Algoritmo Genético (AG).

Se eligió esta técnica debido al gran tamaño del espacio de soluciones, que crece de forma exponencial y hace inviable un recorrido exhaustivo. Además, las restricciones son sencillas de incorporar en la evaluación de cada planificación de tomas.

El AG permite explorar de manera eficiente múltiples asignaciones de tomas a días de grabación simultáneamente. La combinación de selección, cruce y mutación genera nuevas planificaciones de manera continua, aproximando soluciones de bajo coste en un tiempo razonable.

En este diseño se incorporaron además las siguientes estrategias:


*   Elitismo: se conservan las mejores planificaciones de cada generación, evitando que soluciones de buena calidad se pierdan.
*   Criterio de parada: el algoritmo se detiene tran un número fijo de generaciones, suficiente para explorar una buena parte del espacio de soluciones.
* Estudio previo del óptimo: mediante un algoritmo exacto de programación lineal entera se identiicó el valor óptimo (27 actor-días) para validar la calidad de las soluciones aproximadas del AG.

Aunque el AG no garantiza encontrar la solución óptima debido a su naturaleza metaheurística, permite obtener planificaciones cercanas al óptimo con eficiencia, cumpliendo todas las restricciones del problema.



**Mejoras**

Aunque el tiempo de ejecución del algoritmo no es excesivo para este tamaño de problema (alrededor de 1-2 minutos), se ha implementado una mejora clave para optimizarlo:

1. Caché de fitness: La función fitness se calcula para cada individuo cada vez que se evalúa, incluso si ya se ha evaluado antes. Al almacenar los resultados en un diccionario (fitness_cache), se evitan cálculos redundantes y se mejora significativamente el tiempo de ejecución.

2. Elitismo: Se mantiene la conservación de los 3 mejores individuos en cada generación, asegurando que las soluciones de alta calidad no se pierdan mientras se mantiene la coherencia con el algoritmo original.

Estas modificaciones permiten mantener la efectividad del algoritmo original mientras se optimiza el rendimiento disminuyendo el tiempo de ejecución de manera notable.





In [ ]:
import random
import numpy as np

random.seed(42)
np.random.seed(42)

actores_tomas = np.array([
[1,1,1,1,1,0,0,0,0,0],
[0,0,1,1,1,0,0,0,0,0],
[0,1,0,0,1,0,1,0,0,0],
[1,1,0,0,0,0,1,1,0,0],
[0,1,0,1,0,0,0,1,0,0],
[1,1,0,1,1,0,0,0,0,0],
[1,1,0,1,1,0,0,0,0,0],
[1,1,0,0,0,1,0,0,0,0],
[1,1,0,1,0,0,0,0,0,0],
[1,1,0,0,0,1,0,0,1,0],
[1,1,1,0,1,0,0,1,0,0],
[1,1,1,1,0,1,0,0,0,0],
[1,0,0,1,1,0,0,0,0,0],
[1,0,1,0,0,1,0,0,0,0],
[1,1,0,0,0,0,1,0,0,0],
[0,0,0,1,0,0,0,0,0,1],
[1,0,1,0,0,0,0,0,0,0],
[0,0,1,0,0,1,0,0,0,0],
[1,0,1,0,0,0,0,0,0,0],
[1,0,1,1,1,0,0,0,0,0],
[0,0,0,0,0,1,0,1,0,0],
[1,1,1,1,0,0,0,0,0,0],
[1,0,1,0,0,0,0,0,0,0],
[0,0,1,0,0,1,0,0,0,0],
[1,1,0,1,0,0,0,0,0,1],
[1,0,1,0,1,0,0,0,1,0],
[0,0,0,1,1,0,0,0,0,0],
[1,0,0,1,0,0,0,0,0,0],
[1,0,0,0,1,1,0,0,0,0],
[1,0,0,1,0,0,0,0,0,0]
])

NUM_TOMAS = 30
NUM_ACTORES = 10
MAX_TOMAS_DIA = 6
NUM_DIAS = 5

fitness_cache = {}

def fitness(individuo):
    key = tuple(individuo)
    if key not in fitness_cache:
        costo = 0
        penalizacion = 0
        for dia in range(NUM_DIAS):
            indices = [i for i in range(NUM_TOMAS) if individuo[i] == dia]
            penalizacion += abs(len(indices) - MAX_TOMAS_DIA) * 200
            if len(indices) > 0:
                actores_dia = np.zeros(NUM_ACTORES)
                for t in indices:
                    actores_dia = np.logical_or(actores_dia, actores_tomas[t])
                costo += np.sum(actores_dia)
        fitness_cache[key] = costo + penalizacion
    return fitness_cache[key]

def crear_individuo():
    individuo = []
    for d in range(NUM_DIAS):
        individuo += [d]*MAX_TOMAS_DIA
    random.shuffle(individuo)
    return individuo

def seleccion_torneo(poblacion, k=3):
    participantes = random.sample(poblacion, k)
    participantes.sort(key=lambda x: fitness(x))
    return participantes[0]

def cruce(p1, p2):
    punto = random.randint(1, NUM_TOMAS-1)
    h1 = p1[:punto] + p2[punto:]
    h2 = p2[:punto] + p1[punto:]
    return h1, h2

def mutacion(individuo, prob=0.03):
    for i in range(NUM_TOMAS):
        if random.random() < prob:
            j = random.randint(0, NUM_TOMAS-1)
            individuo[i], individuo[j] = individuo[j], individuo[i]
    return individuo

def algoritmo_genetico():
    TAM = 300
    GEN = 800

    poblacion = [crear_individuo() for _ in range(TAM)]

    for g in range(GEN):
        nueva = []
        mejores = sorted(poblacion, key=lambda x: fitness(x))[:3]
        nueva.extend(mejores)
        while len(nueva) < TAM:
            p1 = seleccion_torneo(poblacion)
            p2 = seleccion_torneo(poblacion)
            h1, h2 = cruce(p1, p2)
            nueva.append(mutacion(h1))
            if len(nueva) < TAM:
                nueva.append(mutacion(h2))
        poblacion = nueva
        if g % 50 == 0:
            print(f"Gen {g} - Mejor coste: {fitness(mejores[0])}")
    mejor = min(poblacion, key=lambda x: fitness(x))
    return mejor

sol = algoritmo_genetico()
print("\nMejor solución encontrada:")
print(sol)
print("Costo:", fitness(sol))

Gen 0 - Mejor coste: 35
Gen 50 - Mejor coste: 30
Gen 100 - Mejor coste: 30
Gen 150 - Mejor coste: 29
Gen 200 - Mejor coste: 29
Gen 250 - Mejor coste: 29
Gen 300 - Mejor coste: 29
Gen 350 - Mejor coste: 29
Gen 400 - Mejor coste: 29
Gen 450 - Mejor coste: 29
Gen 500 - Mejor coste: 29
Gen 550 - Mejor coste: 29
Gen 600 - Mejor coste: 29
Gen 650 - Mejor coste: 29
Gen 700 - Mejor coste: 29
Gen 750 - Mejor coste: 29

Mejor solución encontrada:
[4, 4, 2, 1, 2, 4, 4, 0, 0, 1, 1, 0, 2, 3, 1, 2, 0, 3, 3, 4, 1, 0, 3, 3, 2, 1, 2, 4, 3, 0]
Costo: 29
